# 03 暴露與疾病的關聯：2×2 表與推論統計

松柏護理之家退伍軍人症群聚，主管問：「使用淋浴的人，感染風險有沒有比較高？」資深疫調前輩追問：「你怎麼用統計來『證明』？」

這堂課學會：**描述 vs. 推論統計的差異 → 研究設計（世代 vs. 病例對照）→ 2×2 列聯表 → 風險比 (RR) → 勝算比 (OR) → 信賴區間 (CI) → 卡方 / Fisher 檢定 → 多因子彙整與森林圖**。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 描述統計 vs. 推論統計：我們在做什麼？

Ch02 我們用**描述統計**（平均值、次數分布、圖表）整理了資料的樣貌。但主管問的問題是：

> 「淋浴使用和感染**有沒有關聯**？」

這就需要**推論統計（inferential statistics）**——用樣本資料去推論母體中是否真的存在關聯，還是我們看到的差異只是**隨機誤差（chance）**造成的。

### 核心概念

- **虛無假設（H₀）**：淋浴使用與感染互相獨立（無關聯）
- **對立假設（H₁）**：淋浴使用與感染有關聯
- **p-value**：假設 H₀ 為真時，觀察到現有數據（或更極端數據）的機率。p 越小，越有理由拒絕 H₀
- **信賴區間（CI）**：效應量的合理範圍。若 95% CI 不包含「無效果值」（RR=1 或 OR=1），則 p < 0.05

### 研究設計決定你能算什麼——四種常見設計

#### ❶ 世代研究（Cohort Study）——🎬 跟拍紀錄片
先依暴露分兩組（有淋雨 vs. 沒淋雨），一路追蹤誰後來感冒。你知道全部人 → **有完整分母 → 算 RR**。

#### ❷ 病例對照研究（Case-Control Study）——🕵️ 偵探辦案
先找到生病的人（病例），再挑沒病的人（對照），回頭問暴露史。對照組人數是你自己決定的 → **沒有完整分母 → 只能算 OR**。適合：罕見疾病、疫苗效益大規模監測。

#### ❸ 巢式病例對照（Nested Case-Control）——🏠 翻監視器錄影帶
已有世代在追蹤，但逐一檢驗太貴。等有人發病，從世代中挑病例+對照做檢驗。兼具世代的代表性和病例對照的效率。

#### ❹ 配對病例對照（Matched Case-Control）——👯 雙胞胎實驗
每個病例配一個年齡、性別相近的對照，讓干擾因子「抵消」。分析要用條件式邏輯斯迴歸。

| 研究設計 | 抽樣方式 | 可算指標 | 比喻 |
|---------|---------|--------|------|
| 世代研究 | 依暴露分組，追蹤結果 | **RR** | 🎬 跟拍紀錄片 |
| 病例對照 | 依疾病分組，回溯暴露 | **OR** | 🕵️ 偵探辦案 |
| 巢式病例對照 | 從世代中挑病例+對照 | **OR** | 🏠 翻監視器 |
| 配對病例對照 | 1:n 配對，控制干擾 | **OR**（條件式）| 👯 雙胞胎實驗 |

> 🎬 **本次調查** = 回溯性世代研究：280 位住民全部納入，暴露和結果都已知 → 可以直接計算 **RR**。

In [ ]:
# --- Step 1: 資料準備 ---
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import chi2_contingency, fisher_exact
from epi_learning.metrics import risk_ratio, odds_ratio

# -- CJK font setup (避免中文標籤顯示為方框 □□□) --
# matplotlib 預設只認英文字型，中文字會變成「豆腐塊」
# 解法：手動掃描系統字型目錄，註冊所有 CJK（中日韓）字型
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# 動態偵測實際註冊的 CJK 字型名稱（避免 .ttc face 0 陷阱）
_discovered = []
for _entry in fm.fontManager.ttflist:
    _nlower = _entry.name.lower()
    if any(_kw in _nlower for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
_cjk_fonts = list(_discovered)
for _n in _preferred:
    if _n not in _cjk_fonts:
        _cjk_fonts.append(_n)
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# 建立「是否感染」的二元欄位 (0/1)
# clinical_severity == "not_ill" 表示沒有症狀也沒有感染，其餘都算感染
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")
print(f"整體侵襲率：{df['infected'].mean():.1%}")

## Step 2: 建立 2×2 表

我們已有 280 位住民和 `infected` 欄位。現在要整理成 **2×2 列聯表**——流行病學關聯分析的基本資料結構。

> **💡 小提示**：`pd.crosstab()` 的第一個參數變成「列（rows）」，第二個變成「欄（columns）」。提取 a/b/c/d 四格時，先用中文標籤重新命名，就不用記 0 和 1 的對應。

In [ ]:
# --- Step 2: 建立 2×2 表（淋浴 × 感染）---
# pd.crosstab() 的第一個參數 → 列（rows），第二個 → 欄（columns）
# margins=True 自動加上小計列和小計欄
ct_shower = pd.crosstab(
    df["shower_use"], df["infected"],
    margins=True, margins_name="合計",
)
# 重新命名，讓輸出更好讀（原始值 0/1 不直觀）
ct_shower.index = ["未使用淋浴", "使用淋浴", "合計"]
ct_shower.columns = ["未感染", "感染", "合計"]
print(ct_shower)

# ── 提取 2×2 表的四個格子 ──
# crosstab 的列順序取決於原始值排序（0 在 1 前面）
# 重新命名後，我們用中文標籤來提取，就不用記哪個是 0、哪個是 1
a = int(ct_shower.loc["使用淋浴", "感染"])        # a = 暴露＋感染
b = int(ct_shower.loc["使用淋浴", "未感染"])      # b = 暴露＋未感染
c = int(ct_shower.loc["未使用淋浴", "感染"])      # c = 未暴露＋感染
d = int(ct_shower.loc["未使用淋浴", "未感染"])    # d = 未暴露＋未感染

# 分別計算兩組的侵襲率（attack rate）
print(f"\n暴露組（使用淋浴）侵襲率: {a/(a+b):.1%}")
print(f"未暴露組（未使用淋浴）侵襲率: {c/(c+d):.1%}")

In [ ]:
# --- Step 3: 計算 Risk Ratio（風險比）---
rr = risk_ratio(a, a + b, c, c + d)
print(f"淋浴使用 → 感染的 RR = {rr:.3f}")
print(f"  解讀：使用淋浴者的感染風險是未使用者的 {rr:.1f} 倍")
print(f"  RR = 1 → 無關聯 | RR > 1 → 暴露可能增加風險 | RR < 1 → 可能是保護因子")

## RR vs OR — 風險和勝算到底差在哪？

### RR 的白話文 ⚖️

RR = 2 就是「有暴露的人，得病**風險**是沒暴露的 2 倍」。像天平秤重：暴露組的風險放左邊，未暴露組放右邊。

### OR 的白話文 🎰

OR = 3 就是「有暴露的人，得病的**勝算**是沒暴露的 3 倍」。注意：這裡說的是「勝算」不是「風險」！

- **風險 = p**：100 人裡 30 人生病 → 30/100 = 0.3（除以**全部**）
- **勝算 = p/(1−p)**：30 人生病 vs 70 人沒病 → 30/70 ≈ 0.43（生病 ÷ **沒生病**）

### 三個重要觀念

1. **罕見疾病時 OR ≈ RR**（10% 法則）：當 p 很小，1−p ≈ 1，所以 odds ≈ risk → OR ≈ RR。
   🍬 100 顆糖果 3 顆酸的：risk = 3/100 = 3%，odds = 3/97 ≈ 3.1%，幾乎一樣。

2. **侵襲率高時 OR 系統性大於 RR**：本案侵襲率 ~43%，OR 會比 RR 高估 30-50%！不能拿 OR 說「風險是幾倍」。

3. **OR 是邏輯斯迴歸的原生輸出**：模型算的是 log(odds)，所以 exp(β) = OR。Ch06 會教。

### 疫苗效益 VE 怎麼估？

- **世代研究**：VE = 1 − RR（例：RR = 0.2 → VE = 80%）
- **病例對照**：VE = 1 − OR（罕見疾病時 ≈ 1 − RR）
- COVID-19 疫苗很多效益數據就是用 test-negative case-control design 算的

In [ ]:
# --- Step 4: 計算 Odds Ratio（勝算比）---
# 勝算 (odds) = p / (1-p)，和風險 (risk) = p 不同
# OR = (a × d) / (b × c)
or_val = odds_ratio(a, b, c, d)
print(f"淋浴使用 → 感染的 OR = {or_val:.3f}")
print(f"  （相比 RR = {rr:.3f}）")
print(f"\n本資料集侵襲率 = {df['infected'].mean():.1%}（非罕見疾病）")
print(f"→ OR ({or_val:.3f}) 大於 RR ({rr:.3f})，這是預期的")
print(f"→ 疾病罕見時 OR ≈ RR；侵襲率越高，OR 偏離 RR 越多")
print(f"\n⚠️ 本案侵襲率 ~43%，不能拿 OR 來說「風險是幾倍」！")
print(f"   正確：淋浴者感染風險是未使用者的 {rr:.1f} 倍（RR）")
print(f"   錯誤：淋浴者感染風險是未使用者的 {or_val:.1f} 倍（OR）← 高估了！")

## Step 5: 95% 信賴區間 — 為什麼要先取 log？

CI 是新手最容易「看到就頭暈」的部分。關鍵直覺：

1. **原始尺度不對稱**：RR/OR 的範圍是 0 到 ∞，以 1 為中心，左邊只有一小段（0 到 1），右邊卻延伸到無限大
2. **log 轉換到對稱尺度**：取 ln() 後，尺度變成 -∞ 到 +∞，以 0 為中心，可以用常態分布的 ±1.96×SE
3. **exp 轉回去**：把 log 尺度的上下界用 exp() 轉回原始尺度，就是 95% CI

In [ ]:
# --- Step 5: 95% 信賴區間（RR 和 OR）---

# ── RR 的 95% CI：Katz method ──

# (a) 取自然對數：把 RR 從不對稱尺度 (0, ∞) 轉到對稱尺度 (-∞, +∞)
ln_rr = np.log(rr)

# (b) 計算標準誤（SE）：衡量 ln(RR) 估計值的精準度
#     公式來自 Katz（1978），利用 2×2 表的四格推導
se_ln_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))

# (c) 在 log 尺度上 ±1.96 × SE（1.96 是常態分布 95% 的 z 值）
# (d) 用 exp() 轉回原始尺度 → 得到 CI 的上下界
ci_rr_lo = np.exp(ln_rr - 1.96 * se_ln_rr)
ci_rr_hi = np.exp(ln_rr + 1.96 * se_ln_rr)

# ── OR 的 95% CI：Woolf method ──
# 原理相同，只是 SE 的公式不同（直接用 a, b, c, d 的倒數和）
ln_or = np.log(or_val)
se_ln_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_or_lo = np.exp(ln_or - 1.96 * se_ln_or)
ci_or_hi = np.exp(ln_or + 1.96 * se_ln_or)

print("=== 95% 信賴區間比較 ===")
print(f"RR = {rr:.3f} (95% CI: {ci_rr_lo:.3f} – {ci_rr_hi:.3f})")
print(f"OR = {or_val:.3f} (95% CI: {ci_or_lo:.3f} – {ci_or_hi:.3f})")
sig_rr = "顯著" if ci_rr_lo > 1 else "不顯著"
sig_or = "顯著" if ci_or_lo > 1 else "不顯著"
print(f"\nRR 的 CI {'不' if ci_rr_lo <= 1 else ''}包含 1 → {sig_rr}")
print(f"OR 的 CI {'不' if ci_or_lo <= 1 else ''}包含 1 → {sig_or}")

## Step 6: 卡方檢定

卡方檢定的核心邏輯：如果暴露和感染真的「無關」（H₀ 為真），每個格子應該觀察到多少人？實際數字離這個預期有多遠？

- **差距小** → χ² 小 → p 大 → 不顯著（觀察值接近 H₀ 預測）
- **差距大** → χ² 大 → p 小 → 顯著！（觀察值遠離 H₀ 預測）

In [ ]:
# --- Step 6: 卡方檢定 ---
# H₀: 淋浴使用與感染互相獨立（無關聯）
# 邏輯：比較「觀察到的次數」與「假設無關時的期望次數」
contingency = [[a, b], [c, d]]
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"卡方統計量 = {chi2:.3f}")
print(f"自由度 = {dof}")
print(f"p-value = {p:.4f}")
print(f"\n期望值表（H₀ 為真時的預期次數）：")
print(pd.DataFrame(
    expected.round(1),
    index=["使用淋浴", "未使用淋浴"],
    columns=["感染", "未感染"],
))

# 檢查期望值是否都 >= 5
min_expected = expected.min()
print(f"\n最小期望值 = {min_expected:.1f}", end="")
if min_expected >= 5:
    print(" → 滿足卡方檢定前提")
else:
    print(" → < 5，建議改用 Fisher 精確檢定")

In [ ]:
# --- Step 7: Fisher 精確檢定 ---
# 當期望值 < 5 時的替代方案（小樣本更精確）
oddsr_fisher, p_fisher = fisher_exact(contingency)
print(f"Fisher 精確檢定:")
print(f"  OR = {oddsr_fisher:.3f}")
print(f"  p-value = {p_fisher:.4f}")
print(f"\n卡方檢定 p = {p:.4f} vs Fisher p = {p_fisher:.4f}")
print("（此例樣本夠大，兩種檢定結果相近；小樣本時差異會更明顯）")

In [ ]:
# --- Step 8: 第二個暴露因子 — 水療使用 ---
# 用同一套流程分析第二個暴露因子（和 Steps 2–6 完全相同，只換了暴露變數）
# Step 9 會用迴圈自動化這個過程，不用再手動複製貼上
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a2, b2 = int(ct_hydro.loc[1, 1]), int(ct_hydro.loc[1, 0])  # 暴露+感染, 暴露+未感染
c2, d2 = int(ct_hydro.loc[0, 1]), int(ct_hydro.loc[0, 0])  # 未暴露+感染, 未暴露+未感染

# 效應量
rr2 = risk_ratio(a2, a2 + b2, c2, c2 + d2)
or2 = odds_ratio(a2, b2, c2, d2)
chi2_2, p2, _, _ = chi2_contingency([[a2, b2], [c2, d2]])

# RR CI（Katz method，同 Step 5）
ln_rr2 = np.log(rr2)
se_rr2 = np.sqrt(1/a2 - 1/(a2+b2) + 1/c2 - 1/(c2+d2))
ci_rr2_lo = np.exp(ln_rr2 - 1.96 * se_rr2)
ci_rr2_hi = np.exp(ln_rr2 + 1.96 * se_rr2)

# OR CI（Woolf method，同 Step 5）
ln_or2 = np.log(or2)
se_or2 = np.sqrt(1/a2 + 1/b2 + 1/c2 + 1/d2)
ci_or2_lo = np.exp(ln_or2 - 1.96 * se_or2)
ci_or2_hi = np.exp(ln_or2 + 1.96 * se_or2)

print("水療使用 → 感染")
print(f"  RR = {rr2:.3f} (95% CI: {ci_rr2_lo:.3f} – {ci_rr2_hi:.3f})")
print(f"  OR = {or2:.3f} (95% CI: {ci_or2_lo:.3f} – {ci_or2_hi:.3f})")
print(f"  卡方 p-value = {p2:.4f}")

## 森林圖（Forest Plot）是什麼？

**森林圖**是流行病學和實證醫學中最常見的圖表之一，常用於系統性回顧（systematic review）和統合分析（meta-analysis），但在群聚調查中也非常實用——可以**一眼比較多個暴露因子的效應量大小和統計顯著性**。

### 怎麼看森林圖？

- **圓點（●）**：點估計值（本例為 RR）
- **水平線段（─）**：95% 信賴區間
- **虛線（RR = 1）**：無效果線。CI 與虛線交叉 = 不顯著；CI 完全在虛線右側 = 暴露顯著增加風險

下一步我們就來畫一張森林圖，同時比較 8 個危險因子的粗 RR。

In [ ]:
# --- Step 9: 多因子粗效應量彙整表 + 森林圖 ---
# 實際疫調中不會只看一兩個因子。
# 下面的迴圈把 Steps 2–6 系統性地套用到所有候選暴露。

# 列出所有要檢驗的暴露因子（二元 0/1 變數）
factors = [
    "shower_use", "hydrotherapy_use",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
]
# 把「曾經吸菸」轉成二元變數
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)
factors.append("ever_smoker")

# ── 迴圈：對每個因子重複 Steps 2–6 ──
# 每一輪做 5 件事：(a) 建 2×2 表 → (b) 算 RR/OR → (c) 算 CI → (d) 卡方檢定 → (e) 存結果
results = []
for factor in factors:
    # (a) 建立 2×2 表，提取 a, b, c, d
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])   # 暴露＋感染
    b_i = int(ct.loc[1, 0])   # 暴露＋未感染
    c_i = int(ct.loc[0, 1])   # 未暴露＋感染
    d_i = int(ct.loc[0, 0])   # 未暴露＋未感染

    # (b) 效應量：RR 和 OR
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    or_i = odds_ratio(a_i, b_i, c_i, d_i)

    # (c) RR 的 95% CI（Katz method，同 Step 5）
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)

    # (d) 卡方檢定
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])

    # (e) 把這個因子的結果存起來
    results.append({
        "factor": factor,
        "RR": round(rr_i, 3),
        "CI_lower": round(ci_lo, 3),
        "CI_upper": round(ci_hi, 3),
        "OR": round(or_i, 3),
        "p-value": round(p_i, 4),
    })

# 彙整成表格，按 RR 由大到小排序（最可疑的因子排最前面）
rr_table = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== 多因子粗效應量彙整表 ===")
display_df = rr_table.copy()
display_df["95% CI"] = display_df.apply(
    lambda r: f"{r['CI_lower']:.3f}–{r['CI_upper']:.3f}", axis=1
)
print(display_df[["factor", "RR", "95% CI", "OR", "p-value"]].to_string(index=False))

# --- 森林圖（Forest Plot）---
fig, ax = plt.subplots(figsize=(8, 5))
rr_sorted = rr_table.reset_index(drop=True)
y_pos = range(len(rr_sorted))
ax.errorbar(
    rr_sorted["RR"], y_pos,
    xerr=[rr_sorted["RR"] - rr_sorted["CI_lower"],
          rr_sorted["CI_upper"] - rr_sorted["RR"]],
    fmt="o", color="#D97757", ecolor="#6B6B6B", capsize=4, markersize=7,
)
ax.axvline(x=1, color="#6B6B6B", linestyle="--", alpha=0.7, label="RR = 1（無效果）")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(rr_sorted["factor"])
ax.set_xlabel("Risk Ratio (95% CI)")
ax.set_title("各因子粗風險比（Forest Plot）")
ax.legend(loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 小結

| 步驟 | 學到的技能 | Python 工具 |
|------|-----------|------------|
| 2×2 表 | 列聯表建構 | `pd.crosstab()` |
| RR | 風險比計算與解讀 | `risk_ratio()` |
| OR | 勝算比計算，與 RR 比較 | `odds_ratio()` |
| 95% CI | RR 和 OR 的信賴區間 | Katz / Woolf method |
| 卡方檢定 | 獨立性檢定 | `chi2_contingency()` |
| Fisher 精確檢定 | 小樣本替代方案 | `fisher_exact()` |
| 多因子掃描 | 迴圈 + 彙整 + 森林圖 | `for` + `DataFrame` + `matplotlib` |

### 什麼時候用 RR？什麼時候用 OR？

| 情境 | 用哪個 | 為什麼 |
|------|--------|--------|
| 世代研究（如本次群聚調查） | **RR** | 有完整分母，可以直接算風險 |
| 病例對照研究 | **OR** | 沒有完整分母，無法算風險 |
| 邏輯斯迴歸的結果 | **OR** | 模型輸出就是 log-odds |
| 罕見疾病的任何研究設計 | 都可以 | 罕見時 OR ≈ RR |
| 侵襲率高（> 10%） | **RR** | OR 會系統性高估風險倍數 |
| 疫苗效益（世代） | VE = 1−**RR** | 臨床試驗、群聚調查 |
| 疫苗效益（病例對照） | VE = 1−**OR** | 大規模上市後監測 |

### 重要提醒

1. **粗 RR / OR 只是初步線索**。淋浴使用的效應量看起來很高，但可能被**干擾作用（confounding）**影響——就像漢堡看起來很大，但可能是生菜撐出來的。Ch05 會用**分層分析**和 **Mantel-Haenszel 法**來處理。

2. **統計顯著 ≠ 因果關係**。測了 8 個因子，光靠機率就可能有 ~0.4 個「偽陽性」（α=0.05 時）。找到關聯只是起點。

3. **Ch06** 會用**邏輯斯迴歸**同時調整多個因子，算出 adjusted OR（記得：迴歸的原生輸出是 OR，侵襲率高時要小心解讀）。